# Stochastic MPC for LinearLongitudinalB747

This notebook shows how to train and evaluate the stochastic `MPCAgent` on the `LinearLongitudinalB747-v0` environment. The workflow covers data collection under random disturbances, supervised training of the internal dynamics model, and a closed-loop MPC rollout with control-quality metrics.


In [ ]:
import numpy as np
import torch
from tqdm import tqdm

import gymnasium as gym
import matplotlib.pyplot as plt

from tensoraerospace.agent.mpc.stochastic import MPCAgent, Net
from tensoraerospace.signals.random import full_random_signal
from tensoraerospace.signals.standart import unit_step
from tensoraerospace.utils import generate_time_period
from tensoraerospace.benchmark import ControlBenchmark


In [ ]:
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)



In [ ]:
dt = 0.1
simulation_time = 20

tp = generate_time_period(tn=simulation_time, dt=dt)
number_time_steps = len(tp)

reference_signal = np.reshape(
    unit_step(tp, degree=4, time_step=8, output_rad=False),
    (1, -1),
)

initial_state = np.array([0.0, 0.0, 0.0, 0.0], dtype=np.float32)

env = gym.make(
    "LinearLongitudinalB747-v0",
    number_time_steps=number_time_steps,
    initial_state=initial_state,
    reference_signal=reference_signal,
    dt=dt,
)

state, info = env.reset()
print(f"Observation shape: {state.shape}, time steps: {number_time_steps}")


In [ ]:
def tracking_cost(next_state, action, reference_signals=None, step=0):
    """Quadratic tracking cost with mild action penalty."""
    if reference_signals is None:
        target = torch.zeros(next_state.shape[-1], dtype=next_state.dtype)
    else:
        idx = min(step, reference_signals.shape[1] - 1)
        target = torch.as_tensor(reference_signals[:, idx], dtype=next_state.dtype)

    pitch_error = next_state[..., 0] - target[0]
    rate_error = next_state[..., 1]
    action_penalty = 0.01 * torch.norm(action)

    return (pitch_error**2 + 0.25 * rate_error**2).mean() + action_penalty



In [ ]:
system_model = Net(
    num_action=env.action_space.shape[0],
    num_states=env.observation_space.shape[0],
)

agent = MPCAgent(
    gamma=0.99,
    action_dim=env.action_space.shape[0],
    observation_dim=env.observation_space.shape[0],
    model=system_model,
    cost_function=tracking_cost,
    env=env,
    min_max_action_value=(-15.0, 15.0),
    lr=1e-3,
)



In [ ]:
exploration_signal = full_random_signal(
    t0=0.0,
    dt=dt,
    tn=simulation_time,
    sd=(0.3, 0.8),
    sv=(-10.0, 10.0),
)

states, actions, next_states = agent.collect_data(
    num_episodes=35,
    control_exploration_signal=exploration_signal,
)

states = states.reshape(states.shape[0], -1).astype(np.float32)
next_states = next_states.reshape(next_states.shape[0], -1).astype(np.float32)
actions = actions.reshape(-1).astype(np.float32)

print(f"Collected {states.shape[0]} transitions")


In [ ]:
agent.train_model(
    states=states,
    actions=actions,
    next_states=next_states,
    epochs=250,
    batch_size=256,
)



In [ ]:
mpc_states = []
mpc_actions = []

state, _ = env.reset()
mpc_states.append(state.reshape(-1))

reference_array = reference_signal
time_axis = []
max_steps = min(env.number_time_steps - 3, reference_array.shape[1] - 3)

for step in tqdm(range(max_steps), desc="MPC rollout"):
    action, _ = agent.choose_action_ref(
        state.reshape(-1),
        rollout=64,
        horizon=3,
        reference_signals=reference_array,
        step=step,
    )
    next_state, reward, terminated, truncated, _ = env.step(action[0])

    mpc_actions.append(float(action[0]))
    mpc_states.append(next_state.reshape(-1))
    time_axis.append(step * dt)

    state = next_state
    if terminated or truncated:
        break

mpc_states = np.array(mpc_states).reshape(len(mpc_states), -1)
mpc_actions = np.array(mpc_actions)
time_axis = np.array(time_axis)
print(f"Simulated {len(mpc_states)} steps")


In [ ]:
plot_len = min(len(time_axis), reference_signal.shape[1] - 1)

theta_ref = reference_signal[0, :plot_len]
theta_actual = mpc_states[1 : plot_len + 1, 0]
pitch_rate = mpc_states[1 : plot_len + 1, 1]

plt.figure(figsize=(12, 8))
plt.subplot(3, 1, 1)
plt.plot(time_axis[:plot_len], theta_ref, label="Reference θ (deg)")
plt.plot(time_axis[:plot_len], theta_actual, label="MPC θ (deg)")
plt.ylabel("Pitch angle")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 2)
plt.plot(time_axis[:plot_len], pitch_rate, color="tab:orange")
plt.ylabel("Pitch rate (deg/s)")
plt.grid(True, alpha=0.3)

plt.subplot(3, 1, 3)
plt.plot(time_axis[:plot_len], mpc_actions[:plot_len], color="tab:green")
plt.xlabel("Time (s)")
plt.ylabel("Stabilizer deflection (deg)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
benchmark = ControlBenchmark()
metrics = benchmark.becnchmarking_one_step(
    control_signal=theta_ref,
    system_signal=theta_actual,
    signal_val=0.1,
    dt=dt,
)

for name, value in metrics.items():
    print(f"{name:>20s}: {value}")



## Notes

- Adjust `num_episodes`, `epochs`, `rollout`, and `horizon` to trade off sample efficiency and runtime.
- The same workflow applies to other longitudinal aerospace models—only the environment configuration and cost shaping need to change.
